In [3]:
from datetime import datetime


failed_by_ip = {}
failed_by_user = {}
failed_events = []

successful_logins = 0
failed_logins = 0


# Read the log file
with open("securityloginsV2.txt", "r") as file:

    for line in file:

        parts = line.split()

        # Skip empty lines
        if not line.strip():
            continue

        # Get the timestamp
        timestamp_text = parts[0] + " " + parts[1]

        timestamp = datetime.strptime(
            timestamp_text,
            "%Y-%m-%d %H:%M:%S"
        )


        # Successful login
        if "LOGIN_SUCCESS" in line:

            successful_logins += 1


        # Failed login
        elif "LOGIN_FAILED" in line:

            failed_logins += 1

            username = parts[3].split("=")[1]
            ip = parts[4].split("=")[1]


            # Store failed login information
            failed_events.append({
                "time": timestamp,
                "username": username,
                "ip": ip
            })


            # Count failures by IP
            if ip not in failed_by_ip:
                failed_by_ip[ip] = 0

            failed_by_ip[ip] += 1


            # Count failures by username
            if username not in failed_by_user:
                failed_by_user[username] = 0

            failed_by_user[username] += 1


# --------------------------------
# Security Report
# --------------------------------

print("==============================")
print("       SECURITY LOG REPORT")
print("==============================")


print("\nLogin Summary")
print("------------------------------")

print("Successful logins:", successful_logins)
print("Failed logins:", failed_logins)


# --------------------------------
# Failed logins by IP
# --------------------------------

print("\nFailed Logins by IP")
print("------------------------------")

for ip, count in failed_by_ip.items():

    print(ip, ":", count)


# --------------------------------
# Failed logins by username
# --------------------------------

print("\nFailed Logins by Username")
print("------------------------------")

for username, count in failed_by_user.items():

    print(username, ":", count)


# --------------------------------
# Detect rapid failed logins
# --------------------------------

print("\nPotential Brute-Force Activity")
print("------------------------------")


suspicious_ips = set()


for i in range(len(failed_events)):

    current_event = failed_events[i]

    count = 1

    for j in range(i + 1, len(failed_events)):

        next_event = failed_events[j]


        # Only compare events from the same IP
        if next_event["ip"] == current_event["ip"]:

            time_difference = (
                next_event["time"] - current_event["time"]
            ).total_seconds()


            # Check whether events happened within 60 seconds
            if time_difference <= 60:

                count += 1


    if count >= 3:

        suspicious_ips.add(current_event["ip"])


# Display suspicious IPs

if suspicious_ips:

    for ip in suspicious_ips:

        print(
            "⚠ Potential brute-force activity:",
            ip
        )

else:

    print("No potential brute-force activity detected.")

       SECURITY LOG REPORT

Login Summary
------------------------------
Successful logins: 2
Failed logins: 5

Failed Logins by IP
------------------------------
192.0.2.15 : 3
192.0.2.30 : 2

Failed Logins by Username
------------------------------
admin : 3
alex : 2

Potential Brute-Force Activity
------------------------------
⚠ Potential brute-force activity: 192.0.2.15
